In [1]:
import pandas as pd
import requests
import re

# 1. Carregar o dataset
# O PDF sugere usar sep=";" se o arquivo não for vírgula padrão
df = pd.read_csv('carros-fipe.csv', sep=';')

# 2. Remover coluna registro_id
df_original = df.copy() # Backup para gerar arquivos de descarte
if 'registro_id' in df.columns:
    df.drop(columns=['registro_id'], inplace=True)

# 3. Alterar nome do cliente na primeira linha (Requisito: trocar pelo seu nome)
df.at[0, 'cliente'] = 'Seu Nome Aqui'

# --- Funções de Limpeza ---
def limpar_numeros(val):
    if pd.isna(val): return None
    # Remove R$, pontos de milhar e troca vírgula por ponto
    s = str(val).replace('R$', '').replace(' ', '').replace('km', '').strip()
    if ',' in s and '.' in s:
        s = s.replace('.', '').replace(',', '.')
    elif ',' in s:
        s = s.replace(',', '.')
    try:
        return float(s)
    except:
        return None

def limpar_apenas_digitos(val):
    if pd.isna(val): return None
    return re.sub(r'\D', '', str(val))

def limpar_placa(val):
    if pd.isna(val): return None
    return re.sub(r'[^a-zA-Z0-9]', '', str(val))

# 4. Aplicar Limpeza e Formatação
df['marca_id_clean'] = df['marca_id'].astype(str).str.extract(r'(\d+)')[0]
df['modelo_id_clean'] = df['modelo_id'].astype(str).str.extract(r'(\d+)')[0]
df['ano_modelo_clean'] = df['ano_modelo'].str.strip() # Deve estar no padrão aaaa-nn
df['placa_clean'] = df['placa'].apply(limpar_placa)
df['cpf_clean'] = df['cpf'].apply(limpar_apenas_digitos)
df['valor_venda_clean'] = df['valor_venda'].apply(limpar_numeros)
df['taxa_servico_clean'] = df['taxa_servico'].apply(limpar_numeros)

# 5. Identificar problemas e descartar (Campos Obrigatórios: Placa e CPF)
# Também validamos se os IDs necessários para a API existem
is_valid = (
    df['placa_clean'].notna() & 
    df['cpf_clean'].notna() & 
    df['marca_id_clean'].notna() &
    df['modelo_id_clean'].notna() &
    df['ano_modelo_clean'].notna()
)

df_descarte = df_original[~is_valid].copy()
df_working = df[is_valid].copy()

# 6. Consultar Endpoint FIPE
def consultar_fipe(row):
    url = f"https://parallelum.com.br/fipe/api/v1/carros/marcas/{row['marca_id_clean']}/modelos/{row['modelo_id_clean']}/anos/{row['ano_modelo_clean']}"
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            return response.json()
    except:
        pass
    return None

print("Consultando API FIPE... Isso pode levar alguns segundos.")
df_working['fipe_json'] = df_working.apply(consultar_fipe, axis=1)

# 7. Separar descartes do endpoint
is_endpoint_ok = df_working['fipe_json'].notna()
df_descarte_endpoint = df_working[~is_endpoint_ok].copy()
df_final = df_working[is_endpoint_ok].copy()

# 8. Criar novas colunas (preco_fipe, marca, lucro)
def formatar_valor_fipe(json_data):
    # Pega "R$ 88.629,00" -> "88629,00"
    valor_str = json_data.get('Valor', '0')
    return valor_str.replace('R$', '').strip()

df_final['preco_fipe'] = df_final['fipe_json'].apply(formatar_valor_fipe)
df_final['marca'] = df_final['fipe_json'].apply(lambda x: x.get('Marca', ''))

# Calculando Lucro (preco_fipe - valor_venda)
# Converter preco_fipe para float para o cálculo
df_final['preco_fipe_num'] = df_final['preco_fipe'].apply(lambda x: float(x.replace('.', '').replace(',', '.')))
df_final['lucro_num'] = df_final['preco_fipe_num'] - df_final['valor_venda_clean']
df_final['lucro'] = df_final['lucro_num'].apply(lambda x: "{:.2f}".format(x).replace('.', ','))

# 9. Preparar arquivos de saída (Formatação Final)
df_final['valor_total'] = (df_final['valor_venda_clean'] + df_final['taxa_servico_clean']).apply(lambda x: "{:.2f}".format(x).replace('.', ','))
df_final['valor_venda'] = df_final['valor_venda_clean'].apply(lambda x: "{:.2f}".format(x).replace('.', ','))
df_final['taxa_servico'] = df_final['taxa_servico_clean'].apply(lambda x: "{:.2f}".format(x).replace('.', ','))
df_final['km_rodados'] = df_final['km_rodados'].apply(limpar_apenas_digitos)

colunas_saida = [
    'marca_id', 'modelo_id', 'ano_modelo', 'data_venda', 'valor_venda', 
    'taxa_servico', 'valor_total', 'km_rodados', 'placa', 'cliente', 
    'cpf', 'estado', 'preco_fipe', 'marca', 'lucro'
]

# Gerar CSVs
df_final[colunas_saida].to_csv('carros-fipe-saida.csv', index=False, sep=';')
df_descarte.to_csv('carros-fipe-descarte.csv', index=False, sep=';')
df_descarte_endpoint.to_csv('carros-fipe-descarte-endpoint.csv', index=False, sep=';')

# 10. Pivotamento (total_por_marca.csv)
# Extrair o ano do ano_modelo (aaaa-nn)
df_final['ano_ext'] = df_final['ano_modelo'].str[:4]
pivot = pd.pivot_table(
    df_final, 
    values='valor_venda_clean', 
    index='marca', 
    columns='ano_ext', 
    aggfunc='sum', 
    fill_value=0
)
pivot.to_csv('total_por_marca.csv', sep=';')

print("Processamento concluído com sucesso!")

Consultando API FIPE... Isso pode levar alguns segundos.
Processamento concluído com sucesso!
